KPI 3 - Error Rate


Loading libararies 

In [6]:
pip install statsmodels


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.5 MB 4.9 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.5 MB 4.8 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.5 MB 4.1 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.5 MB 4.0 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.5 MB 4.2 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.5 MB 4.2 MB/s eta 0:00:02
   -------------------------- ------------- 6.3/9.5 MB 4.3 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.5 MB 4.2 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.5 MB 3.9 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.5 MB 3.7 MB/s eta 0:00:01
   ---------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\Desmond\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest

Loading data sets 

In [8]:
data_path = "../Data"
exp = pd.read_csv(f"{data_path}/df_final_experiment_clients_cleaned.csv")
funnel = pd.read_csv(f"{data_path}/web_funnel_sorted.csv")

Confrimation Check 

In [9]:
print(funnel.shape)
print(funnel.columns.tolist())
print(funnel["process_step"].unique())  # confirm step names
print(exp["group_test"].unique())       # confirm "Test" / "Control" labels

(744641, 6)
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'source']
<StringArray>
['step_3', 'start', 'step_1', 'step_2', 'confirm']
Length: 5, dtype: str
<StringArray>
['test', 'control', nan]
Length: 3, dtype: str


Hypothesis:

The New  UI has the same or more errors than the  Old UI 


The New  UI has fewer errors than than Old UI

- H₀: error_rate(Test) ≥ error_rate(Control)   
- H₁: error_rate(Test) < error_rate(Control)   

Error  is defiend as a backward step in the funnel, movingin backwards from step 2 to step 1 indicating user cofusion or friction 

Detecting Errors 

 Sorting  by client and timestamp to preserve step order


In [ ]:
funnel_sorted = funnel.sort_values(["client_id", "date_time"]).copy()



Mapping  steps to numeric order


In [ ]:
step_order = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
funnel_sorted["step_num"] = funnel_sorted["process_step"].map(step_order)


<StringArray>
['step_3', 'start', 'step_1', 'step_2', 'confirm']
Length: 5, dtype: str

In [20]:
print(exp["group_test"].unique())
print(exp["group_test"].value_counts())

<StringArray>
['test', 'control', nan]
Length: 3, dtype: str
group_test
test       26968
control    23532
Name: count, dtype: int64


In [21]:
print(client_errors.shape)
print(client_errors["group_test"].value_counts())
print(client_errors["had_error"].value_counts())

(70609, 3)
group_test
test       26968
control    23532
Name: count, dtype: int64
had_error
False    45114
True     25495
Name: count, dtype: int64


Flag a row as an error if step went backward vs previous step for that client


In [ ]:
funnel_sorted["prev_step"] = funnel_sorted.groupby("client_id")["step_num"].shift(1)
funnel_sorted["is_error"] = funnel_sorted["step_num"] < funnel_sorted["prev_step"]

 One row per client: did they make at least one error?

In [13]:
client_errors = (
    funnel_sorted.groupby("client_id")["is_error"]
    .any()
    .reset_index()
    .rename(columns={"is_error": "had_error"})
)





Merge with experiment group labels

In [14]:
client_errors = client_errors.merge(
    exp[["client_id", "group_test"]], on="client_id", how="inner"
)
print(client_errors["group_test"].value_counts())
print(client_errors["had_error"].value_counts())

group_test
test       26968
control    23532
Name: count, dtype: int64
had_error
False    45114
True     25495
Name: count, dtype: int64


Group by Error rates 

In [15]:
error_rates = (
    client_errors.groupby("group_test")["had_error"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "errors", "count": "total", "mean": "error_rate"})
)

print(error_rates)



            errors  total  error_rate
group_test                           
control       8089  23532    0.343745
test         10195  26968    0.378041


Testsing 

In [23]:
from statsmodels.stats.proportion import proportions_ztest

test_grp    = client_errors[client_errors["group_test"] == "test"]
control_grp = client_errors[client_errors["group_test"] == "control"]

count = [test_grp["had_error"].sum(), control_grp["had_error"].sum()]
nobs  = [len(test_grp), len(control_grp)]


# alternative='smaller' → H₁: Test error rate < Control error rate
z_stat, p_value = proportions_ztest(count, nobs, alternative="smaller")

print(f"Z-statistic : {z_stat:.4f}")
print(f"P-value     : {p_value:.4f}")
print(f"Alpha       : 0.05")
print()
if p_value < 0.05:
    print("Reject H₀ — Test group has a significantly LOWER error rate.")
else:
    print("Fail to reject H₀ — No significant reduction in error rate.")


Z-statistic : 7.9996
P-value     : 1.0000
Alpha       : 0.05

Fail to reject H₀ — No significant reduction in error rate.


Final Conclusion 

The result supports H₀ — the Test group had a higher error rate than Control, meaning the new UI did not reduce errors and may have actually increased user friction.